# Plan D1: DAPT afro-xlmr-large on Kenya corpus

Continue MLM on `out/dapt_corpus.parquet` (76,197 tweets). Free T4 runtime.

- Every cell is safe to re-run: training auto-resumes from Drive checkpoints.
- Run the smoke cell first on a fresh session - it prints the real CUDA per-step time.
- The full run checkpoints every 500 steps; expect ~3-4h and 1-2 disconnect/resume cycles.
- If OOM at batch 8: add `--grad-checkpoint`, then `--batch-size 4 --grad-accum 16`,
  then fall back to `--model Davlan/afro-xlmr-base` (whole plan drops to base size).

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%pip install -q "transformers>=4.48" "accelerate>=1.0" "scikit-learn>=1.4" "pandas>=2" "pyarrow>=18" matplotlib

In [ ]:
# smoke: sample mode (1k texts, 20 steps, out/dapt-sample) - ~2 min on T4
!cd /content/drive/MyDrive/Colab/hatespeech-finetune && python 09_dapt.py

In [ ]:
# full run: re-run this cell after any disconnect, it resumes automatically
!cd /content/drive/MyDrive/Colab/hatespeech-finetune && python 09_dapt.py --full

In [ ]:
import json
log = json.load(open('/content/drive/MyDrive/Colab/hatespeech-finetune/out/dapt-afro-xlmr/dapt_log.json'))
print(f"stock loss {log['baseline']['eval_loss']:.4f} ppl {log['baseline']['ppl']:.1f}")
print(f"dapt  loss {log['final_eval_loss']:.4f} ppl {log['final_ppl']:.1f}")
print(f"gain {log['gain']:.4f} - gate (>= 0.10): {'MET' if log['gain'] >= 0.10 else 'NOT MET'}")

In [ ]:
# push DAPT model to private HF repo (needs HF_TOKEN in Colab secrets)
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
!pip install -q huggingface_hub
!cd /content/drive/MyDrive/Colab/hatespeech-finetune && python 10_push_hf.py --model-dir out/dapt-afro-xlmr --repo-id kenya-dapt-afroxlmr